# 04. Model Prediksi Risiko Pembatalan (Cancel Risk Model)

Notebook ini melatih model klasifikasi Machine Learning untuk memprediksi probabilitas pembatalan pesanan saat checkout (pre-dispatch):
- Menggunakan fitur pre-dispatch murni tanpa kebocoran data (*data leakage*)
- Evaluasi model baseline (Logistic Regression) dan model pohon (Random Forest, Gradient Boosting)
- Penanganan ketidakseimbangan kelas via `class_weight="balanced"`
- Perhitungan metrik evaluasi: ROC-AUC, PR-AUC, F1-Score, Confusion Matrix
- Ekstraksi Feature Importance dan arah koefisien risiko
- Ekspor `cancel_risk_summary.json` (tabel risiko multidimensi & metrik)
- Ekspor `cancel_risk_model.json` (bobot model parametrik untuk inferensi langsung di backend Node.js tanpa runtime Python)


In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score, confusion_matrix
)

DATA_PATH = "../data/processed/orders_ml.csv"
BACKEND_ML_DIR = "../../backend/src/data/ml"
os.makedirs(BACKEND_ML_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dataset transaksi siap model: {df.shape[0]} baris, {df.shape[1]} kolom.")


Dataset transaksi siap model: 42356 baris, 22 kolom.


## 1. Seleksi Fitur Pre-Dispatch & Pembagian Data (Train/Test Split)
Hanya menggunakan informasi yang tersedia saat pesanan dibuat. Seluruh fitur logistik pasca pengiriman (`Courier Status`, `is_delivered`, `fulfilled-by`) dikecualikan untuk menjamin integritas prediksi.

In [2]:
cat_features = ["Category", "Size", "is_fba", "is_expedited", "ship_state_clean", "is_b2b", "has_promotion"]
num_features = ["unit_price", "qty_imputed", "day_of_week", "is_weekend", "day_of_month"]
target = "is_cancelled"

X = df[cat_features + num_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Data Latih: {X_train.shape[0]} baris | Data Uji: {X_test.shape[0]} baris")
print(f"Distribusi Target Train: {y_train.value_counts().to_dict()}")


Data Latih: 33884 baris | Data Uji: 8472 baris
Distribusi Target Train: {0: 28979, 1: 4905}


## 2. Preprocessing & Feature Encoding
- Standarisasi fitur numerik dengan `StandardScaler`\n- Encoding fitur kategorikal dengan `OneHotEncoder(handle_unknown="ignore")`

In [3]:
scaler = StandardScaler()
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_num = scaler.fit_transform(X_train[num_features])
X_test_num = scaler.transform(X_test[num_features])

X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat = ohe.transform(X_test[cat_features])

cat_feature_names = ohe.get_feature_names_out(cat_features).tolist()
all_feature_names = num_features + cat_feature_names

X_train_enc = np.hstack([X_train_num, X_train_cat])
X_test_enc = np.hstack([X_test_num, X_test_cat])
print(f"Total dimensi matriks fitur terenkode: {X_train_enc.shape[1]} kolom.")


Total dimensi matriks fitur terenkode: 69 kolom.


## 3. Pelatihan Model Klasifikasi Risiko & Evaluasi Metrik
Melatih Logistic Regression (sebagai basis inferensi cepat di backend) dan Random Forest (analisis interaksi fitur nonlinear).

In [4]:
# Logistic Regression
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train_enc, y_train)
lr_probs = lr.predict_proba(X_test_enc)[:, 1]
lr_preds = (lr_probs >= 0.50).astype(int)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_split=10, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train_enc, y_train)
rf_probs = rf.predict_proba(X_test_enc)[:, 1]

# Evaluasi
roc_auc = float(roc_auc_score(y_test, lr_probs))
rf_auc = float(roc_auc_score(y_test, rf_probs))
pr_auc = float(average_precision_score(y_test, lr_probs))
acc = float(accuracy_score(y_test, lr_preds))
f1_macro = float(f1_score(y_test, lr_preds, average="macro"))
f1_c1 = float(f1_score(y_test, lr_preds, pos_label=1))
prec_c1 = float(precision_score(y_test, lr_preds, pos_label=1))
rec_c1 = float(recall_score(y_test, lr_preds, pos_label=1))
cm = confusion_matrix(y_test, lr_preds).tolist()

print(f"Logistic Regression ROC-AUC: {roc_auc:.4f} (Ambang batas target: >= 0.65) -> LULUS!")
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f} | Akurasi: {acc:.4f} | Recall: {rec_c1:.4f} | F1-Score: {f1_c1:.4f}")
print("Confusion Matrix [[TN, FP], [FN, TP]]:", cm)


Logistic Regression ROC-AUC: 0.9315 (Ambang batas target: >= 0.65) -> LULUS!
Random Forest ROC-AUC: 0.9411
PR-AUC: 0.7697 | Akurasi: 0.7855 | Recall: 0.9796 | F1-Score: 0.5695
Confusion Matrix [[TN, FP], [FN, TP]]: [[5453, 1792], [25, 1202]]


## 4. Ekstraksi Feature Importance & Arah Koefisien Risiko

In [5]:
rf_importances = rf.feature_importances_
top_indices = np.argsort(rf_importances)[::-1][:15]

top_features_list = []
for idx in top_indices:
    fname = all_feature_names[idx]
    weight = float(rf_importances[idx])
    coef_val = float(lr.coef_[0][idx])
    direction = "Menaikkan Risiko" if coef_val > 0 else "Menurunkan Risiko"
    top_features_list.append({
        "feature": fname,
        "importance": round(weight, 4),
        "logistic_weight": round(coef_val, 4),
        "direction": direction
    })

feat_df = pd.DataFrame(top_features_list)
print("Top 10 Fitur Paling Berpengaruh terhadap Pembatalan:")
print(feat_df.head(10).to_string(index=False))


Top 10 Fitur Paling Berpengaruh terhadap Pembatalan:
                     feature  importance  logistic_weight         direction
             has_promotion_1      0.5061          -3.3630 Menurunkan Risiko
             has_promotion_0      0.3815           3.1123  Menaikkan Risiko
                    is_fba_1      0.0154          -2.1859 Menurunkan Risiko
                  unit_price      0.0143           0.0055  Menaikkan Risiko
                    is_fba_0      0.0118           1.9352  Menaikkan Risiko
              is_expedited_0      0.0104          -1.0213 Menurunkan Risiko
              is_expedited_1      0.0080           0.7705  Menaikkan Risiko
                day_of_month      0.0076           0.0410  Menaikkan Risiko
ship_state_clean_Maharashtra      0.0060          -0.9785 Menurunkan Risiko
                 day_of_week      0.0045           0.0176  Menaikkan Risiko


## 5. Pembuatan Tabel Risiko Multidimensi & Ekspor cancel_risk_summary.json

In [6]:
def calc_risk_table(dataframe, group_col):
    agg = dataframe.groupby(group_col).agg(
        total_orders=("is_cancelled", "count"),
        cancelled_orders=("is_cancelled", "sum"),
        cancel_rate=("is_cancelled", "mean")
    ).reset_index()
    
    result = {}
    for _, r in agg.iterrows():
        rate = float(r["cancel_rate"])
        level = "Rendah" if rate < 0.15 else ("Sedang" if rate <= 0.20 else "Tinggi")
        result[str(r[group_col])] = {
            "total_orders": int(r["total_orders"]),
            "cancelled_orders": int(r["cancelled_orders"]),
            "cancel_rate": round(rate, 4),
            "risk_level": level
        }
    return result

# 1. By Category
risk_by_cat = calc_risk_table(df, "Category")

# 2. By Region (State)
risk_by_state = calc_risk_table(df[df["ship_state_clean"] != "Unknown"], "ship_state_clean")

# 3. By Fulfilment
risk_by_fulfilment = calc_risk_table(df, "Fulfilment")

# 4. By Price Tier
df["price_tier"] = pd.cut(
    df["unit_price"],
    bins=[0, 300, 600, 1000, 100000],
    labels=["< INR 300", "INR 300 - 600", "INR 600 - 1000", "> INR 1000"]
)
risk_by_price = calc_risk_table(df, "price_tier")

# 5. By Day of Week
risk_by_day = calc_risk_table(df, "day_name")

summary_payload = {
    "model_metrics": {
        "model_name": "Logistic Regression (Class-Balanced) & Random Forest Classifier",
        "roc_auc": round(roc_auc, 4),
        "rf_roc_auc": round(rf_auc, 4),
        "pr_auc": round(pr_auc, 4),
        "accuracy": round(acc, 4),
        "f1_macro": round(f1_macro, 4),
        "f1_class_1": round(f1_c1, 4),
        "precision_class_1": round(prec_c1, 4),
        "recall_class_1": round(rec_c1, 4),
        "confusion_matrix": cm,
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    },
    "risk_thresholds": {
        "low": 0.20,
        "medium": 0.40,
        "high": 0.40
    },
    "top_feature_importances": top_features_list[:10],
    "risk_by_category": risk_by_cat,
    "risk_by_region": risk_by_state,
    "risk_by_fulfilment": risk_by_fulfilment,
    "risk_by_price_tier": risk_by_price,
    "risk_by_day_of_week": risk_by_day
}

summary_file = os.path.join(BACKEND_ML_DIR, "cancel_risk_summary.json")
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)
print(f"cancel_risk_summary.json berhasil disimpan ke {summary_file}")


cancel_risk_summary.json berhasil disimpan ke ../../backend/src/data/ml\cancel_risk_summary.json

## 6. Ekspor cancel_risk_model.json untuk Inferensi di JavaScript Backend
Mengekspor bobot model linear yang dapat dievaluasi secara instan oleh backend Node.js tanpa memerlukan dependensi Python runtime saat produksi.

In [7]:
# Ekstrak intercept dan koefisien per fitur
intercept = float(lr.intercept_[0])
coef_map = {}
for fname, cval in zip(all_feature_names, lr.coef_[0]):
    coef_map[fname] = round(float(cval), 5)

num_stats = {}
for idx, col in enumerate(num_features):
    num_stats[col] = {
        "mean": round(float(scaler.mean_[idx]), 4),
        "std": round(float(scaler.scale_[idx]), 4)
    }

cat_categories = {}
for col, cats in zip(cat_features, ohe.categories_):
    cat_categories[col] = [str(c) for c in cats]

model_payload = {
    "model_type": "LogisticRegression",
    "algorithm": "sigmoid(intercept + sum(w_num * (x - mean)/std) + sum(w_cat * is_cat))",
    "intercept": round(intercept, 5),
    "coefficients": coef_map,
    "numerical_features": num_stats,
    "categorical_features": cat_categories,
    "thresholds": {
        "low": 0.20,
        "medium": 0.40
    },
    "javascript_helper": (
        "function predictCancelRisk(features) {\n"
        "  let logit = intercept;\n"
        "  for (const [col, stats] of Object.entries(numerical_features)) {\n"
        "    const val = features[col] ?? stats.mean;\n"
        "    const z = (val - stats.mean) / stats.std;\n"
        "    logit += (coefficients[col] ?? 0) * z;\n"
        "  }\n"
        "  for (const col of Object.keys(categorical_features)) {\n"
        "    const val = String(features[col]);\n"
        "    const key = `${col}_${val}`;\n"
        "    logit += coefficients[key] ?? 0;\n"
        "  }\n"
        "  return 1 / (1 + Math.exp(-logit));\n"
        "}"
    )
}

model_file = os.path.join(BACKEND_ML_DIR, "cancel_risk_model.json")
with open(model_file, "w", encoding="utf-8") as f:
    json.dump(model_payload, f, indent=2)
print(f"cancel_risk_model.json berhasil disimpan ke {model_file}")


cancel_risk_model.json berhasil disimpan ke ../../backend/src/data/ml\cancel_risk_model.json
